# NFL passing-yard bets with Nori

Two paths: **A. Replay the saved 42 selections** (downloadable ledger, no model download); **B. Rebuild checkpoint-history features and rerun Nori** (requires the research workspace, saved input tables, quote cache, and original model checkpoint).

Path A reproduces the reported arithmetic, not the feature pipeline, model inference, or market selection. The 2025 strategy was refined retrospectively. Quotes do not prove fills or available quantity. Nothing here places orders.

Sources: [nflverse](https://github.com/nflverse/nflverse-data/releases), [Kalshi API](https://docs.kalshi.com/), [Nori](https://github.com/Synthefy/synthefy-nori).

## A. Replay the frozen selections
The compact ledger is downloaded from the companion website. For offline use, download the JSON once and set DATA_SOURCE to its local path. Source artifact SHA-256: `fce6e5811237af7990e54b4097c350267f119eab14a51dad6b4e18b2b1c02b92`. YES pays $1 when the threshold is met; NO pays $1 otherwise. Each entry buys one contract.

In [ ]:
import json, csv
from decimal import Decimal, ROUND_CEILING
from pathlib import Path

from urllib.request import urlopen

# Set DATA_SOURCE to a downloaded JSON path for offline use.
# The website asset must be deployed before this default URL works.
DATA_SOURCE = 'https://www.synthefy.com/data/nori-nfl-passing-yards/selected-bets-2025.json'
if DATA_SOURCE.startswith('https://'):
    with urlopen(DATA_SOURCE, timeout=30) as response:
        trades = json.load(response)
else:
    trades = json.loads(Path(DATA_SOURCE).read_text())
assert len(trades) == 42
assert len({r["game_id"] for r in trades}) == 39
assert len({(r["game_id"], r["actual_qb_name"]) for r in trades}) == 42
trades[:3]

## Costs and returns
The fee assumption is ceil(0.07 × price × (1 − price) × 100) / 100 per contract. The extra allowance is a sensitivity parameter, not evidence of a fill. ROI is profit divided by purchase price plus fees and allowance, not return on starting bankroll. Selections stay fixed when costs change.

In [ ]:
def replay(rows, allowance_cents=5):
    ledger = []
    for r in rows:
        price = Decimal(str(r['yes_ask'])) if r['side'] == 'yes' else 1 - Decimal(str(r['yes_bid']))
        fee = (Decimal('0.07') * price * (1-price) * 100).to_integral_value(rounding=ROUND_CEILING) / 100
        assert abs(float(fee) - r['fee']) < 1e-9
        cost = price + fee + Decimal(str(allowance_cents))/100
        pnl = Decimal(str(r['settlement_value'])) - cost
        ledger.append({**r, 'price': float(price), 'spread_cents': round(100*(r['yes_ask']-r['yes_bid']), 2), 'cost': float(cost), 'pnl': float(pnl)})
    cost = sum(r['cost'] for r in ledger)
    pnl = sum(r['pnl'] for r in ledger)
    return ledger, {'entries': len(ledger), 'cost': round(cost, 2), 'pnl': round(pnl, 2), 'roi_percent': 100*pnl/cost if cost else None}

ledger, result = replay(trades)
assert result['cost'] == 19.34 and result['pnl'] == 4.66
assert abs(result['roi_percent'] - 24.095139607) < 1e-7
print(result)
for cents in (0, 5, 10):
    print(cents, replay(trades, cents)[1])
for checkpoint in ('q1', 'halftime'):
    print(checkpoint, replay([r for r in trades if r['selected_horizon'] == checkpoint])[1])

In [ ]:
with Path('nori_nfl_selected_bets.csv').open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(ledger[0]))
    writer.writeheader()
    writer.writerows(ledger)
print('Saved nori_nfl_selected_bets.csv')

## B. Rebuild features and rerun predictions (optional)
This path requires access to the Nori research monorepo and its cached inputs. It is not a standalone public-data reproduction. Install the experiment requirements into a separate environment first. Set the paths below; leave RUN_RESEARCH=False for the portable replay. No GPU workload runs by default.

The existing builder adds QB checkpoint means over three and eight prior games, separately for Q1 and halftime. It excludes the current week. The Nori runner uses the existing offense/defense/context inputs, usage features, prior-only correlation pruning at 0.75, and weekly context updates. It scores 2024 and 2025. Use the original checkpoint for numerical reproduction; a newer public checkpoint may differ. The runner refuses to overwrite existing results.

In [ ]:
RUN_RESEARCH = False
EXPERIMENT_DIR = Path('path/to/nori-monorepo/experiments/2026-08-25-nfl-qb-passing-yards')
MODEL_PATH = Path('path/to/original/nori.pt')
DEVICE = 'cpu'

if RUN_RESEARCH:
    import sys, subprocess
    import polars as pl
    required = ['checkpoint_history_ablation.py', 'checkpoint_history_report.py',
                'outputs/qb_game_live_q1_v2_season_context_features.parquet',
                'outputs/qb_game_live_flow_v2_season_context_features.parquet']
    for name in required:
        if not (EXPERIMENT_DIR / name).is_file():
            raise FileNotFoundError(EXPERIMENT_DIR / name)
    if not MODEL_PATH.is_file():
        raise FileNotFoundError(MODEL_PATH)
    sys.path.insert(0, str(EXPERIMENT_DIR.resolve()))
    from checkpoint_history_ablation import add_checkpoint_history
    for horizon, filename in [('q1', 'qb_game_live_q1_v2_season_context_features.parquet'),
                              ('halftime', 'qb_game_live_flow_v2_season_context_features.parquet')]:
        rows, features, _ = add_checkpoint_history(pl.read_parquet(EXPERIMENT_DIR / 'outputs' / filename))
        print(horizon, rows.select(list(features)).head())
        subprocess.run([sys.executable, 'checkpoint_history_ablation.py', '--horizon', horizon,
                        '--arm', 'history', '--device', DEVICE, '--model-path', str(MODEL_PATH.resolve())],
                       cwd=EXPERIMENT_DIR, check=True)
    from checkpoint_history_report import replay as replay_full_strategy
    q1 = pl.read_parquet(EXPERIMENT_DIR / 'outputs/checkpoint_history_history_q1_2025_predictions.parquet')
    ht = pl.read_parquet(EXPERIMENT_DIR / 'outputs/checkpoint_history_history_halftime_2025_predictions.parquet')
    decisions, summary = replay_full_strategy(q1, ht)
    print(summary)
else:
    print('Research rerun skipped. Path A used saved predictions/selections.')

## Adapting the strategy
The full replay checks Q1 first. Only if no Q1 bet qualifies does it consider halftime, requiring the same-side probability on the same line to be at least its Q1 value. One entry per quarterback-game; hold to final settlement. The original quote filter allows at most five-minute age and a 10¢ spread, and the edge threshold is 10¢ before the reporting allowance. Replaying only the downloaded selected rows cannot evaluate alternative selection rules.

For new models, build one row per QB/game/checkpoint from timestamped data. Freeze feature selection and strategy settings before a new evaluation season. Historical player and team identity encodings must be fit on context rows only. Do not use final-game statistics as in-game inputs. Report MAE, RMSE, quantile loss and P10–P90 coverage on all eligible predictions, separately from betting ROI. Never substitute later favorable prices or infer liquidity from a trade print.